# 00 · Setup Environment

**Layer:** foundation &nbsp;|&nbsp; **Stage:** setup

Creates the three medallion databases and pins the Spark settings the rest of
the pipeline relies on. This is the only notebook that performs DDL against
the metastore, so every downstream stage can assume its target database
already exists.

| Database | Layer | Holds |
| --- | --- | --- |
| `banking_bronze` | Bronze | raw landed records, verbatim, plus ingest metadata |
| `banking_silver` | Silver | cleansed, deduplicated, conformed entities |
| `banking_gold` | Gold | business-ready aggregates serving reporting |


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("medallion-banking-setup")
    .enableHiveSupport()
    .getOrCreate()
)

# The cluster default of 200 shuffle partitions is far too high for the
# volumes this demo moves - it produces hundreds of empty tasks per stage.
spark.conf.set("spark.sql.shuffle.partitions", "16")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

print("Spark version :", spark.version)
print("Shuffle parts :", spark.conf.get("spark.sql.shuffle.partitions"))

In [ ]:
from datetime import date

# Run parameters. When driven by a Yeedu pipeline these are supplied as task
# parameters; the literals below are the interactive-development defaults.
RUN_DATE   = "2026-09-03"
BATCH_ID   = f"batch_{RUN_DATE.replace('-', '')}"
DQ_THRESHOLD = 0.98          # minimum pass ratio for a layer to be promoted

BRONZE_DB = "banking_bronze"
SILVER_DB = "banking_silver"
GOLD_DB   = "banking_gold"

print(f"run_date     = {RUN_DATE}")
print(f"batch_id     = {BATCH_ID}")
print(f"dq_threshold = {DQ_THRESHOLD}")

In [ ]:
for db in (BRONZE_DB, SILVER_DB, GOLD_DB):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db}")
    print(f"ready: {db}")

In [ ]:
# The audit table is created up front so the publish stage can append to it
# even on the very first run of the pipeline.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD_DB}.pipeline_audit_log (
        batch_id      STRING,
        run_date      STRING,
        stage         STRING,
        table_name    STRING,
        row_count     BIGINT,
        status        STRING,
        logged_at     TIMESTAMP
    )
    USING parquet
""")

spark.sql("SHOW DATABASES").show(truncate=False)

In [ ]:
print("=" * 60)
print("SETUP COMPLETE")
print(f"  databases : {BRONZE_DB}, {SILVER_DB}, {GOLD_DB}")
print(f"  batch_id  : {BATCH_ID}")
print("=" * 60)